# Lecture 03 — Logistic Regression for Predictive Analytics (Python)
**Term:** Fall 2025  
**Week/Topic:** Lecture 04 — Logistic Regression  
**Instructor:** Dr. Bushaj  

### What we'll cover (detected from this notebook)
- Probability ↔ odds ↔ log-odds (logit); logistic/sigmoid response
- Maximum Likelihood Estimation (MLE) overview
- `sklearn.linear_model.LogisticRegression` (fit, predict, probabilities)
- `statsmodels` GLM (Binomial) for inference (coefficients / ORs)
- Train/validation/test split; stratification where applicable
- ROC curve and AUC
- Regularization: L1/L2/Elastic Net
- Encode categorical variables (one-hot/dummies)
- Gains and Lift analysis for targeting
- Class imbalance strategies (e.g., `class_weight`)
- Interpreting coefficients in log-odds / odds ratios


## Read Data

In [ ]:
my_drive_path = "/content/drive/MyDrive/SUNY/Class Material/2025 Fall/MSA550/Organization Example/data/"

In [ ]:
bank_df = pd.read_csv(my_drive_path + 'UniversalBank.csv')

In [ ]:
bank_df

In [ ]:
bank_df.drop(columns=['ID', 'ZIP Code'], inplace=True)
bank_df.columns = [c.replace(' ', '_') for c in bank_df.columns]

In [ ]:
bank_df.columns

## Bank Data - Model with a Single Predictor


In [ ]:
# Define predictors and outcome
predictors = ['Income']  # We will first use only 'Income' as the predictor
outcome = 'Personal_Loan'

In [ ]:
y = bank_df[outcome]
X = bank_df[predictors]

In [ ]:
X

In [ ]:
# Partition data into training and validation sets
train_X, valid_X, train_y, valid_y = train_test_split(X, y, test_size=0.4, random_state=1)
print(f"Training set size: {train_X.shape}, Validation set size: {valid_X.shape}")


## Build a Model

In [ ]:
logit_reg = LogisticRegression()


# We are creating a model with default strategies. We can define many different paramters in each of the models we define.
# FOR EXAMPLE:
"""
logit_reg = LogisticRegression(
    penalty="l2",    # Regularization type ('l2' = Ridge regularization, 'l1' = Lasso regularization, 'elasticnet' = combination of both, 'none' = no regularization)
    C=1e42,          # Inverse of regularization strength. Smaller values imply stronger regularization (default is 1.0). A very large value like 1e42 essentially disables regularization.
    solver='liblinear',  # Optimization algorithm to use. Options:
                         #   - 'liblinear': Good for small datasets, supports L1 and L2 penalties, binary classification
                         #   - 'lbfgs': Recommended for multiclass problems (more than 2 classes), supports L2 and none penalties, efficient with large datasets
                         #   - 'newton-cg': Like 'lbfgs', supports L2 and none penalties, good for large datasets
                         #   - 'sag': Stochastic average gradient descent, good for large datasets, supports L2 and none penalties
                         #   - 'saga': Extension of 'sag', supports L1, L2, and elasticnet penalties, also good for large datasets
    max_iter=100,    # Maximum number of iterations the solver will take before stopping. Default is 100. Increase this if your model isn't converging.
    random_state=1,  # Seed for random number generation, used for reproducibility of results.
    class_weight=None,  # Weights associated with classes. Options:
                       #   - None: No class weighting (default)
                       #   - 'balanced': Adjusts weights inversely proportional to class frequencies (useful for imbalanced datasets)
                       #   - dict: Manually specify weights for each class
    multi_class='auto',  # Determines how to handle multiclass classification. Options:
                         #   - 'auto': Chooses 'ovr' (one-vs-rest) for binary classification or small datasets, 'multinomial' for larger datasets
                         #   - 'ovr': One-vs-rest, fits one classifier per class
                         #   - 'multinomial': Fits a single classifier for all classes, works best with 'lbfgs', 'newton-cg', or 'saga' solvers
    verbose=0,        # Controls verbosity of the solver. Set to >0 for more detailed logging of the optimization process.
    tol=1e-4,         # Tolerance for stopping criteria. If the change in the loss function is smaller than `tol`, the algorithm stops.
    fit_intercept=True,  # Whether to fit an intercept term (bias). Default is True. Set to False if your data is already centered.
    intercept_scaling=1, # Only used when `solver='liblinear'`. It scales the intercept when fit_intercept is True.
    n_jobs=None       # Number of CPU cores used for parallel computation. Only relevant for solvers that support parallelism ('sag', 'saga', 'lbfgs').
)

"""




logit_reg.fit(train_X, train_y)

In [ ]:
print(f'Intercept: {logit_reg.intercept_[0]}')
print(f'Coefficient: {logit_reg.coef_[0][0]}')

## Evaluate Model Performance

In [ ]:
# Evaluate model performance
print("Training Set Performance:")
classificationSummary(train_y, logit_reg.predict(train_X))
print("-------------------------------------------------")
print("Validation Set Performance:")
classificationSummary(valid_y, logit_reg.predict(valid_X))

# Calculate and print AIC for the model
print(f'AIC: {AIC_score(valid_y, logit_reg.predict(valid_X), df=len(predictors) + 1)}')

In [ ]:
# Predicting class labels and probabilities for the validation set
# 'predict' returns the predicted class labels (0 or 1)
# 'predict_proba' returns the predicted probabilities for each class (0 and 1)

logit_reg_pred = logit_reg.predict(valid_X)           # Predicted class labels (0 or 1)
logit_reg_proba = logit_reg.predict_proba(valid_X)    # Predicted probabilities for class 0 and class 1


In [ ]:
# Create a DataFrame to store the actual values, predicted probabilities, and predicted class labels
logit_result = pd.DataFrame({
    'actual': valid_y,                               # Actual target variable (0 or 1)
    'p(0)': logit_reg_proba[:, 0],                   # Probability of class 0
    'p(1)': logit_reg_proba[:, 1],                   # Probability of class 1
    'predicted': logit_reg_pred                      # Predicted class label
})

In [ ]:
# display four different cases
interestingCases = [2764, 932, 2721, 702]
print(logit_result.loc[interestingCases])

In [ ]:
# Confusion matrices provide insight into how well the classifier is performing
# They show TP (True Positive), TN (True Negative), FP (False Positive), and FN (False Negative) counts.
# This helps evaluate the model's performance at the default cutoff of 0.5

print("Confusion Matrix for Training Set")
classificationSummary(train_y, logit_reg.predict(train_X))   # Classification summary for training data

print("\nConfusion Matrix for Validation Set")
classificationSummary(valid_y, logit_reg.predict(valid_X))   # Classification summary for validation data